# Ingesting HLS granules into a hybrid Icechunk/Iceberg store

This notebook walks the whole `icechest` demo pipeline a cell at a time,
against the real NASA MAAP HLS STAC-geoparquet archive and the real LP DAAC
COGs it references. It selects a handful of granule records, rewrites their
asset hrefs to the S3 URLs a virtual reference will point at, reads each
COG's overview pyramid depth, builds the Zarr convention attributes that
make a written array self-describing, and then commits a batch of granules
-- their tabular metadata and their virtual arrays together -- into a local
store in one snapshot.

It needs a `~/.netrc` entry for `urs.earthdata.nasa.gov` and AWS credentials
that can read `s3://nasa-maap-data-store`.

In [ ]:
# 1. Open the archive and look at what a record holds.
from icechest.demo.source import open_archive, select_granules

archive = open_archive()
rows = select_granules(archive, limit=5)
rows.select(["id", "datetime", "proj:epsg", "proj:shape", "proj:transform"]).to_pandas()

In [ ]:
# 2. The asset hrefs, rewritten to the URLs the store will record.
from icechest.demo.assets import asset_urls

row = rows.to_pylist()[0]
asset_urls(row)

In [ ]:
# 3. How many resolution levels this COG carries.
from icechest.demo.credentials import object_store_registry
from icechest.demo.tiff import parse_ifds
from icechest.demo.virtualize import read_header

registry = object_store_registry()
parse_ifds(read_header(asset_urls(row)["B04"], registry))

In [ ]:
# 4. The convention attributes, from the record alone.
from icechest.demo.conventions import granule_attrs

granule_attrs(
    epsg=row["proj:epsg"],
    shape=row["proj:shape"],
    transform=row["proj:transform"],
    levels=5,
)

In [ ]:
# 5. Open a local store and declare the table from the archive's own schema.
from pathlib import Path
from icechest.demo.store import ensure_table, open_store

repo = open_store(Path("hls-demo-store"))
ensure_table(repo, archive.schema())
repo.read("main").pointers

In [ ]:
# 6. Ingest the batch. One commit carries the rows and every virtual array.
from icechest.demo.ingest import ingest_batch

result = ingest_batch(repo, rows, registry=registry)
result.snapshot_id, len(result.committed), result.skipped

In [ ]:
# 7. Query the metadata, then follow a row to its pixels.
snap = repo.read("main")
table = snap.table("granules").scan().to_arrow()
print(table.select(["id", "array_path"]).to_pandas())

granule = result.committed[0]
snap.group[f"{granule}/B04/multiscales/0"][:16, :16]